# Khai báo các hàm phụ trợ tương tự như xử lí Dataset

In [45]:
import os
import pandas as pd
import numpy as np
import cv2
from matplotlib import pyplot as plt
import mediapipe as mp

## Trích xuất keyframes

In [46]:
def getGrayFramesAndFrames(frame_buffer):
    gray_frames = [cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) for frame in frame_buffer]
    return gray_frames

In [47]:
def calculate_histogram_differences(gray_frames):
    HDiffs = []
    # Duyệt qua tất cả các frame trừ frame cuối vì thường là frame không có giá trị
    for i in range(0, len(gray_frames)-1):
        if (i == 0):
        #[gray_frames[i]]: grayframe thứ i, [0] kênh chứa độ sáng, None: tính toàn bộ ảnh, không dùng mask, [256]: 256 bins, 0 <=[0, 256]: giá trị pixel < 256
            hist_curr = cv2.calcHist([gray_frames[i]], [0], None, [256], [0, 256])
            continue
        # Gán frame ở vòng lặp trước cho hist_prev và tính lại hist_curr
        hist_prev = hist_curr
        hist_curr = cv2.calcHist([gray_frames[i]], [0], None, [256], [0, 256])

        Hdiff = np.sum(np.abs(hist_prev - hist_curr))
        HDiffs.append(Hdiff)
    return HDiffs

In [48]:
def Extract_key_frames(frames):
    gray_frames = getGrayFramesAndFrames(frames)
    HDiffs = calculate_histogram_differences(gray_frames)
    mean = np.mean(HDiffs)
    std = np.std(HDiffs)
    threshold = mean + std
    # Chọn keyframes dựa trên ngưỡng
    keyframes = []
    for i in range(len(HDiffs)):
        if HDiffs[i] > threshold:
            # Lấy frame i+1 vì Hdiffs[i] là độ khác biệt của frame thứ i +1 với thứ i
            keyframes.append(frames[i+1])
    return keyframes

## Trích xuất landmarks

In [49]:
def euclidean_distance(v1, v2):
    return np.sqrt((float(v1[0]) - float(v2[0])) ** 2 + (float(v1[1]) - float(v2[1])) ** 2)

In [50]:
def extract_pose_landmarks(rgb_frame, mp_pose):
    pose_results = mp_pose.process(rgb_frame)
    pose_landmarks = []

    if pose_results.pose_landmarks:
        for i, lm in enumerate(pose_results.pose_landmarks.landmark):
            if i < 17 and i not in [7, 8]:  # Loại bỏ từ hông trở xuống và 2 tai
                pose_landmarks.append((lm.x, lm.y))

    return pose_landmarks

In [51]:
def classify_hands (pose_landmarks, hand_landmarks):
    left_wrist_pose  = pose_landmarks[13]  # Cổ tay trái từ Pose là 15 trừ đi 2 tai đã lượt bỏ nên idx = 13
    right_wrist_pose = pose_landmarks[14]  # Cổ tay phải từ Pose là 16 trừ đi 2 tai đã lượt bỏ nên idx = 14
    wrist = hand_landmarks[0] 
    

    dleft = euclidean_distance(left_wrist_pose, wrist)
    dright = euclidean_distance(right_wrist_pose, wrist)
   
    if(dleft < dright):
        return "Left"
    else:
        return "Right"

In [52]:
def extract_hand_landmarks(rgb_frame, mp_hands, pose_landmarks):
    hands_results = mp_hands.process(rgb_frame)
    left_hand_landmarks = []
    right_hand_landmarks = []
    

    if hands_results.multi_hand_landmarks and hands_results.multi_handedness:
        for hand_landmarks, handedness in zip(hands_results.multi_hand_landmarks, hands_results.multi_handedness):
            # Không sử dụng hướng tay của mediapipe vì độ chính xác thấp và thường ngược hướng
            landmarks = [(lm.x, lm.y) for lm in hand_landmarks.landmark]
            if classify_hands(pose_landmarks, landmarks) == "Right":
                right_hand_landmarks = landmarks
            else:
                left_hand_landmarks = landmarks   



        
    
    return left_hand_landmarks, right_hand_landmarks

In [53]:
def extract_landmarks(frames):
    mp_pose = mp.solutions.pose.Pose(static_image_mode=True)
    mp_hands = mp.solutions.hands.Hands(static_image_mode=True, max_num_hands=2, min_detection_confidence=0.1)
    
    landmarks_dict = {}
    
    for idx, frame in enumerate(frames):
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        pose_landmarks = extract_pose_landmarks(rgb_frame, mp_pose)
        left_hand_landmarks, right_hand_landmarks = extract_hand_landmarks(rgb_frame, mp_hands ,pose_landmarks)
        
        landmarks_dict[idx] = {
            "pose": pose_landmarks,
            "left": left_hand_landmarks,
            "right": right_hand_landmarks
        }
    #giải phóng tài nguyên
    mp_pose.close()
    mp_hands.close()
    return landmarks_dict

In [54]:
def filter_invalid_landmarks(landmarks_dict):
    validated = {}
    for landmarks_idx, landmarks_data in landmarks_dict.items():
        validated_landmarks = {}
        # Nếu không có pose thì bỏ qua frame
        if  not landmarks_data["pose"]:
            continue
        for part in ["pose", "right", "left"]:
            # Nếu không có dữ liệu cho phần này, gán mặc định
            if part not in landmarks_data or not landmarks_data[part]:
                if part in ["right", "left"]:
                    validated_landmarks[part] = [(0.0, 0.0)] * 21
            else:
                processed_points = []
                for point in landmarks_data[part]:
                    x = float(point[0])
                    y = float(point[1])
                    if not (0.0 <= x <= 1.0 and 0.0 <= y <= 1.0):
                        x, y = 0.0, 0.0
                    processed_points.append((x, y))
                validated_landmarks[part] = processed_points
        validated[landmarks_idx] = validated_landmarks
    return validated

## Chuẩn hóa với sign space

In [55]:
def calculate_head_unit(pose_landmarks):

    left_eye, right_eye = pose_landmarks[3], pose_landmarks[6]
    # mép ngoài 2 mắt
    head_unit = euclidean_distance(left_eye,right_eye)
    return head_unit


In [56]:
def calculate_sign_space(pose_landmarks):
    head_unit = calculate_head_unit(pose_landmarks)

    nose = pose_landmarks[0]
   
   
    width = 7 * head_unit
    # height = 9.5 * head_unit 
    
    center_x, center_y = nose

    x1 = center_x - width / 2

    y1 = center_y - 1.5 * head_unit  # Cạnh trên cách 1,5 head unit
    x2 = center_x + width / 2
    # y2 = min(1.0 , int(center_y + 7.5 * head_unit))
    y2 = center_y + 8 * head_unit 
    return [x1, y1, x2, y2]

In [57]:
def calculate_all_sign_space(landmarks_dict):
    sign_spaces = {}
    for idx, landmarks_data in landmarks_dict.items():
        sign_spaces[idx] = calculate_sign_space(landmarks_data["pose"])
    return sign_spaces

In [58]:
def normalize_landmarks_to_sign_space(landmarks_dict, sign_spaces):
    
    normalized = {}
    for landmarks_idx, landmarks_data in landmarks_dict.items():
        Xmin, Ymin, Xmax, Ymax = sign_spaces[landmarks_idx]
        w = Xmax - Xmin
        h = Ymax - Ymin
        normalized_landmarks = {}
        for part in ["pose", "right", "left"]:
            processed_points = []
            for point in landmarks_data[part]:
                x = float(point[0])
                y = float(point[1])
                if x != 0.0 and y != 0.0:
                   x = (x - Xmin) / w
                   y = (y - Ymin) / h
                processed_points.append((x, y))
            normalized_landmarks[part] = processed_points
        normalized[landmarks_idx] = normalized_landmarks
    return normalized

# Khai báo Model


In [59]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import KFold
from torch.utils.data import DataLoader, random_split, Dataset
from torch.nn.utils.rnn import pad_sequence
from torch.optim.lr_scheduler import ReduceLROnPlateau
from Model import Sign2PoseTransformer


In [60]:
INPUT_DIM = 114
D_MODEL = 256
NUM_CLASSES = 100
NUM_HEADS = 8
ENC_LAYERS = 4
DEC_LAYERS = 4
FEEDFORWARD_DIM = 2048
DROPOUT = 0.3

# Tạo model
asl_model = Sign2PoseTransformer(
    input_dim=INPUT_DIM,
    d_model=D_MODEL,
    nhead=NUM_HEADS,
    num_encoder_layers=ENC_LAYERS,
    num_decoder_layers=DEC_LAYERS,
    dim_feedforward=FEEDFORWARD_DIM,
    dropout=DROPOUT,
    num_classes=NUM_CLASSES
)
state_dict = torch.load('../Model/best_model_9-4.pth', map_location=torch.device('cpu'))
asl_model.load_state_dict(state_dict)
asl_model.eval() 

Sign2PoseTransformer(
  (input_proj): Linear(in_features=114, out_features=256, bias=True)
  (pos_encoder): PositionalEncoding(
    (dropout): Dropout(p=0.3, inplace=False)
  )
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (linear1): Linear(in_features=256, out_features=2048, bias=True)
        (dropout): Dropout(p=0.3, inplace=False)
        (linear2): Linear(in_features=2048, out_features=256, bias=True)
        (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.3, inplace=False)
        (dropout2): Dropout(p=0.3, inplace=False)
      )
    )
  )
  (decoder): TransformerDecoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerDecoderLayer(
        (self_a

In [ ]:

def preprocess_sequence(landmarks_dict):
    sequence = []

    # Sắp xếp theo thứ tự thời gian
    for frame_id in sorted(landmarks_dict.keys(), key=lambda x: int(x)):
        frame_data = landmarks_dict[frame_id]
        # Trường hợp thiếu right/left/pose thì thêm toàn 0
        pose = frame_data.get('pose', [(0.0, 0.0)] * 33)
        right = frame_data.get('right', [(0.0, 0.0)] * 21)
        left = frame_data.get('left', [(0.0, 0.0)] * 21)

        all_landmarks = pose + right + left
        flattened = [coord for point in all_landmarks for coord in point]
        sequence.append(flattened)

    # Trả về PyTorch tensor
    return torch.tensor(sequence, dtype=torch.float32)

# Xử lí từ livecam

In [62]:
import cv2
import mediapipe as mp
import time
import threading
from queue import Queue
import requests


In [63]:
# Khởi tạo pose từ MediaPipe
mp_pose = mp.solutions.pose
pose = mp_pose.Pose()
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

# Queue để model xử lý
action_queue = Queue()
recognized_words_queue = Queue()
# Cấu hình timeout
MAX_IDLE_FRAMES = 5
MIN_IDLE_FRAMES = 15
MAX_NO_NEW_WORD_FRAMES = 120 #4s 
MIN_WORDS_FOR_SENTENCE = 3  # Tối thiểu số từ để tạo câu






In [64]:
# === XỬ LÝ TRẠNG THÁI TAY ===
def check_action_state(pose_landmarks):
    left_wrist = pose_landmarks.landmark[15]
    right_wrist = pose_landmarks.landmark[16]
    left_hip = pose_landmarks.landmark[23]
    right_hip = pose_landmarks.landmark[24]

    hips_y = min(left_hip.y, right_hip.y)

    # Nếu cả 2 tay thấp hơn hông hoặc nằm ngoài khung hình
    if ((left_wrist.y * 1.1 > hips_y or left_wrist.y > 1) and
        (right_wrist.y * 1.1 > hips_y or right_wrist.y > 1)):
        return "IDLE"
    
    # Nếu 1 trong 2 tay cao hơn hông và trong khung hình
    if ((left_wrist.y < hips_y and left_wrist.y < 1) or
        (right_wrist.y < hips_y and right_wrist.y < 1)):
        return "ACTIVE"
    
    return "IDLE"

In [65]:
index_to_word = {}

with open(r"..\data\raw\dataset\wlasl_class_list.txt", "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():  # bỏ dòng trống
            parts = line.strip().split("\t")  # tách theo tab
            if len(parts) == 2:
                index = int(parts[0])
                word = parts[1]
                index_to_word[index] = word

                

In [66]:
def send_message(ip_address, message):
    try:
        # URL endpoint gửi tin nhắn
        url = f"http://{ip_address}/send"
        
        # Gửi request POST
        response = requests.post(url, data={"message": message})
        
        # Kiểm tra kết quả
        if response.status_code == 200:
            print(f"✅ Gửi tin nhắn thành công: {message}")
            return True
        else:
            print(f"❌ Lỗi gửi tin nhắn. Mã lỗi: {response.status_code}")
            return False
    
    except requests.exceptions.RequestException as e:
        print(f"❌ Lỗi kết nối: {e}")
        return False

In [67]:
# === THREAD XỬ LÝ MODEL ===
def model_worker():
    while True:
        segment = action_queue.get()
        if segment is None:
            break
        print(f"[MODEL] Nhận đoạn {len(segment)} frames để xử lý.")
        keyframes = Extract_key_frames(segment)
        landmarks_dict = extract_landmarks(keyframes)
        filtered_landmarks = filter_invalid_landmarks(landmarks_dict)
        sign_spaces = calculate_all_sign_space(filtered_landmarks)
        normalized_landmarks = normalize_landmarks_to_sign_space(filtered_landmarks, sign_spaces)
        # print(normalized_landmarks)
        # TODO: Gọi model tại đây để nhận diện thủ ngữ


        input_tensor = preprocess_sequence(normalized_landmarks)  # (frames, features)
        input_tensor = input_tensor.unsqueeze(0)  # (1, frames, features)

        # Dự đoán
        with torch.no_grad():
            output = asl_model(input_tensor)  # (1, num_gloss)
            predicted_idx = torch.argmax(output, dim=1).item()
            predict = index_to_word[predicted_idx]
            print("Predicted:", predict)
            # send_message("IP", predict)
            recognized_words_queue.put(predict)
        print(f"[MODEL] Đã xử lý xong đoạn {len(segment)} frames.")


        action_queue.task_done()

# Khởi chạy thread xử lý model
threading.Thread(target=model_worker, daemon=True).start()


[MODEL] Nhận đoạn 19 frames để xử lý.
Predicted: basketball
[MODEL] Đã xử lý xong đoạn 19 frames.
[MODEL] Nhận đoạn 17 frames để xử lý.
Predicted: dark
[MODEL] Đã xử lý xong đoạn 17 frames.
[MODEL] Nhận đoạn 24 frames để xử lý.
Predicted: orange
[MODEL] Đã xử lý xong đoạn 24 frames.
[MODEL] Nhận đoạn 29 frames để xử lý.
Predicted: basketball
[MODEL] Đã xử lý xong đoạn 29 frames.
[MODEL] Nhận đoạn 18 frames để xử lý.
Predicted: basketball
[MODEL] Đã xử lý xong đoạn 18 frames.
[MODEL] Nhận đoạn 19 frames để xử lý.
Predicted: basketball
[MODEL] Đã xử lý xong đoạn 19 frames.
[MODEL] Nhận đoạn 23 frames để xử lý.
Predicted: basketball
[MODEL] Đã xử lý xong đoạn 23 frames.
[MODEL] Nhận đoạn 21 frames để xử lý.
Predicted: basketball
[MODEL] Đã xử lý xong đoạn 21 frames.
[MODEL] Nhận đoạn 16 frames để xử lý.
Predicted: basketball
[MODEL] Đã xử lý xong đoạn 16 frames.
[MODEL] Nhận đoạn 20 frames để xử lý.
Predicted: basketball
[MODEL] Đã xử lý xong đoạn 20 frames.
[MODEL] Nhận đoạn 18 frames để

In [68]:
import json

with open("config.json", "r") as config_file:
    config = json.load(config_file)
    GEMINI_API_KEY = config["GEMINI_API_KEY"]


In [ ]:
from google import genai
from google.genai import types

def translate_glosses(glosses: str, api_key: str) -> str:
    client = genai.Client(api_key=api_key)

    prompt = f"Please translate the following ASL sign glosses into a complete and meaningful English sentence: {glosses}"
    system_instruction = (
        "You are an expert in translating ASL (American Sign Language) glosses "
        + " into grammatically correct and natural-sounding English sentences."
        + " Output only one sentence. If you can't generate a sentence, just return: can't generate."
    )

    response = client.models.generate_content(
        model="gemini-2.0-flash",
        contents=[prompt],
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            max_output_tokens=80,
        ),
    )

    return response.text if response.text else "Can't generate sentence"

In [71]:

def sentence_worker():
    if recognized_words_queue.qsize() >= MIN_WORDS_FOR_SENTENCE:
        print("[SENTENCE] Không có từ mới trong thời gian dài. Tạo câu...")
        words = ""
        while not recognized_words_queue.empty():
            word = recognized_words_queue.get()
            words += word + " "
        sentence = translate_glosses(words, GEMINI_API_KEY)
        print("[SENTENCE]", sentence)
        # send_message(IP,sentence)
    else:
        print("[SENTENCE] Không đủ từ để tạo câu.")


In [72]:
url = 'http://192.168.187.31:81/stream'  # Thay đúng IP 


In [73]:
# === XỬ LÝ CAMERA ===
# cap = cv2.VideoCapture(url)
cap = cv2.VideoCapture(0)


frame_buffer = []
idle_count = 0
is_action_active = False

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        print("Không thể đọc frame từ camera.")
        break

    fps = cap.get(cv2.CAP_PROP_FPS)

    image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = pose.process(image_rgb)

    if results.pose_landmarks:
        state = check_action_state(results.pose_landmarks)

        # ACTIVE → đang thực hiện động tác
        if state == "ACTIVE":
            frame_buffer.append(frame.copy())
            idle_count = 0
            if not is_action_active:
                print("[INFO] → BẮT ĐẦU động tác")
                is_action_active = True

        # IDLE → nghỉ tay
        elif state == "IDLE":
            idle_count += 1
            if is_action_active:
                
                frame_buffer.append(frame.copy())

                if idle_count >= MAX_IDLE_FRAMES:
                    print("[INFO] → KẾT THÚC động tác, đẩy vào model")

                    # Cắt 5 frame IDLE cuối
                    valid_segment = frame_buffer[:-MAX_IDLE_FRAMES] if len(frame_buffer) > MAX_IDLE_FRAMES else []

                    if valid_segment:
                        if len(valid_segment) > MIN_IDLE_FRAMES:
                        
                            action_queue.put(valid_segment)
                            print(f"[DEBUG] → Đoạn động tác: {len(valid_segment)} frames.")

                    # Reset
                    frame_buffer.clear()
                    idle_count = 0
                    is_action_active = False
            elif idle_count == MAX_NO_NEW_WORD_FRAMES:
                print("[INFO] → Không có động tác mới trong thời gian dài, tạo câu")
                threading.Thread(target=sentence_worker, daemon=True).start()         
        # Vẽ trạng thái lên màn hình
        frame = cv2.flip(frame, 1)
        cv2.putText(frame, f"FPS: {fps:.2f}", (10, 70),
            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)
        
        cv2.putText(frame, f"State: {state}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0) if state == "ACTIVE" else (0, 0, 255), 2)
        
    mp_drawing.draw_landmarks(
            frame,
            results.pose_landmarks,
            mp_pose.POSE_CONNECTIONS,
            landmark_drawing_spec=mp_drawing_styles.get_default_pose_landmarks_style())
    cv2.imshow("Sign Language Capture", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

[INFO] → BẮT ĐẦU động tác
[INFO] → KẾT THÚC động tác, đẩy vào model
[DEBUG] → Đoạn động tác: 19 frames.
[INFO] → BẮT ĐẦU động tác
[INFO] → KẾT THÚC động tác, đẩy vào model
[DEBUG] → Đoạn động tác: 17 frames.
[INFO] → BẮT ĐẦU động tác
[INFO] → KẾT THÚC động tác, đẩy vào model
[INFO] → Không có động tác mới trong thời gian dài, tạo câu
[SENTENCE] Không đủ từ để tạo câu.
[INFO] → BẮT ĐẦU động tác
[INFO] → KẾT THÚC động tác, đẩy vào model
[DEBUG] → Đoạn động tác: 24 frames.
[INFO] → Không có động tác mới trong thời gian dài, tạo câu
[SENTENCE] Không có từ mới trong thời gian dài. Tạo câu...
[SENTENCE] The basketball is dark orange.

[INFO] → BẮT ĐẦU động tác
[INFO] → KẾT THÚC động tác, đẩy vào model
[DEBUG] → Đoạn động tác: 29 frames.
[INFO] → BẮT ĐẦU động tác
[INFO] → KẾT THÚC động tác, đẩy vào model
[DEBUG] → Đoạn động tác: 18 frames.
[INFO] → BẮT ĐẦU động tác
[INFO] → KẾT THÚC động tác, đẩy vào model
[DEBUG] → Đoạn động tác: 19 frames.
[INFO] → Không có động tác mới trong thời gian dài,

In [74]:
# Dọn dẹp
cap.release()
cv2.destroyAllWindows()
action_queue.put(None)  # Dừng thread